Panel App – How to Run on NCAR JupyterHub (HPC)

1) Log into JupyterHub:
   https://jupyterhub.hpc.ucar.edu/

2) Open a Terminal in JupyterLab

3) Activate the conda environment (if needed):
   conda activate <YOUR_CONDA_ENV>

4) Navigate to this file’s directory

5) Start the Panel server (keep this terminal running):
   panel serve panel_app.py --address 127.0.0.1 --port 5006 \
       --allow-websocket-origin="jupyterhub.hpc.ucar.edu"

6) Open the app in your browser (same JupyterHub session):
   https://jupyterhub.hpc.ucar.edu/stable/user/<USER_NAME>/proxy/5006/panel_app

7) Stop the app:
   Go back to the terminal and press Ctrl+C

In [1]:
import xarray as xr
from pathlib import Path
from era5_plot import plot_png, NETCDF_FILE, VAR_NAME, TIME_NAME, LEV_NAME, PRES_NAME, LAT_NAME, LON_NAME
import panel as pn
import param

DATA_DIR /glade/derecho/scratch/pearse/CREDIT/RAW_OUTPUT/panelTest/
NEWEST /glade/derecho/scratch/pearse/CREDIT/RAW_OUTPUT/panelTest/foo_sym23


In [2]:
from datasetSelector2 import DatasetBrowser
from metadata import DatasetMetadata
from datasetPlot import DatasetPlot2
from commandRunner import CommandRunner
from inferenceTab import InferenceTab

In [3]:
#pn.extension(raw_css=[Path("static/styles.css").read_text()])
pn.extension('modal')

In [4]:
#DATA_DIR = Path("/Users/vapor/Data/model_predict")
DATA_DIR = Path("/glade/derecho/scratch/pearse/CREDIT/RAW_OUTPUT/panelTest/")
DATASET_METADATA = {}

In [5]:
def scan_datasets():
    for d in DATA_DIR.iterdir():
        if d.is_dir():
            nc_file = f"{d}/*.nc"
            with xr.open_mfdataset(nc_file, engine="netcdf4", autoclose=True) as ds:
                DATASET_METADATA[d.name] = {
                    "ntime": len(ds.time),
                    "nlev": len(ds.get(LEV_NAME, [])),
                    "nplev": int(ds.sizes[PRES_NAME]),
                    "nlat": int(ds.sizes[LAT_NAME]),
                    "nlon": int(ds.sizes[LON_NAME]),
                    "stime": str(ds.time.values[0].astype("datetime64[s]")),
                    "etime": str(ds.time.values[-1].astype("datetime64[s]")),
                    "vars2d": [v for v in ds.data_vars if len(ds[v].dims) <= 3],
                    "vars3d": [v for v in ds.data_vars if len(ds[v].dims) > 3]
                }

In [6]:
scan_datasets()

/glade/derecho/scratch/pearse/tmp/ipykernel_37061/3129563253.py:5: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  with xr.open_mfdataset(nc_file, engine="netcdf4", autoclose=True) as ds:
/glade/derecho/scratch/pearse/tmp/ipykernel_37061/3129563253.py:5: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  with xr.open_

In [7]:
def available_datasets():
    return sorted(
        d.name for d in DATA_DIR.iterdir()
        if d.is_dir()
    )

In [8]:
browser = DatasetBrowser(datasets=available_datasets())

['2026-01-26T00Z', '2026-01-28T06Z', 'foo', 'foo_sym', 'foo_sym2', 'foo_sym23']


In [9]:
@pn.depends(browser.param.checked_items)
def plot_grid(datasets):

    if not datasets:
        return pn.pane.Markdown("### Select one or more datasets")

    plots = [
        DatasetPlot2(dataset=ds, metadata=DATASET_METADATA).panel()
        for ds in datasets
    ]

    return pn.GridBox(
        *plots,
        ncols=2,
        sizing_mode=None,
        css_classes=["plot-grid"],
        styles={
            "grid-auto-rows": "min-content",
            "align-items": "start"
        },
    )

In [10]:
metadata = DatasetMetadata(metadata=DATASET_METADATA)
def sync_active_dataset(event):
    metadata.active_key = event.new
browser.param.watch(sync_active_dataset, 'active_dataset')

Watcher(inst=DatasetBrowser(active_dataset='', checked_items=[], name='DatasetBrowser00124'), cls=<class 'datasetSelector2.DatasetBrowser'>, fn=<function sync_active_dataset at 0x15267458a0c0>, mode='args', onlychanged=True, parameter_names=('active_dataset',), what='value', queued=False, precedence=0)

In [11]:
sidebar = pn.Column(
    "## Datasets",
    browser.panel,
    metadata.panel,
    width=250
)

In [12]:
main = pn.Column(
    plot_grid,
    sizing_mode="stretch_width",
    css_classes=["main-content"]    
)

In [13]:
vis = pn.Row(
    sidebar,
    main,
    sizing_mode="stretch_both",
    styles={"height" : "100vh"}
)

In [14]:
template = pn.template.BootstrapTemplate(title="Forecast Studio")

In [15]:
commandRunner = CommandRunner()
inference = pn.Column(
    commandRunner.panel()
)

In [16]:
inferenceTab = InferenceTab()

tabs = pn.Tabs(
    ("Visualization", vis),
    ("Inference", inference),
    ("Inference2", inferenceTab.panel())
)
template.main[:] = [
    pn.Column(tabs, sizing_mode="stretch_both")
]
template.servable()

BootstrapTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] BootstrapTemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-23253602267456] Column(sizing_mode='stretch_both')
        [0] Tabs
            [0] Row(sizing_mode='stretch_both', styles={'height': '100vh'})
                [0] Column(width=250)
                    [0] Markdown(str)
                    [1] Column(max_height=600, scroll=True, sizing_mode='stretch_width', styles={'border': '1px solid #ddd...})
                        [0] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='2026-01-26T00Z', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                        [1] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='2026-01-28T06Z', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                        [2] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='foo', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                        [3] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='foo_sym', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                        [4] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='foo_sym2', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                        [5] Row(sizing_mode='stretch_width', styles={'background': 'transparen...})
                            [0] Checkbox(align='center', margin=(10, 0, 2, 0), width=10)
                            [1] Button(align='center', name='foo_sym23', sizing_mode='stretch_width', stylesheets=['\n                .bk-bt...])
                    [2] ParamMethod(method, _pane=HTML, defer_load=False)
                [1] Column(css_classes=['main-content'], sizing_mode='stretch_width')
                    [0] ParamFunction(function, _pane=Markdown, defer_load=False)
            [1] Column
                [0] Column(min_height=300, sizing_mode='stretch_width')
                    [0] Card(collapsed=True, title='Command Runner')
                        [0] TextInput(name='Command input', placeholder='credit_rollout_realtime -..., sizing_mode='stretch_width')
                        [1] Row
                            [0] Button(button_type='primary', height=40, min_width=120, name='Run Command', sizing_mode='fixed')
                            [1] Button(height=40, name='Clear Output', sizing_mode='fixed', width=120)
                            [2] LoadingSpinner(color='primary', height=30, visible=False, width=30)
                        [2] StaticText(name='Output log', sizing_mode='stretch_width', value="<pre style='background:#f...)
            [2] Column(min_height=300, sizing_mode='stretch_width')
                [0] Row(sizing_mode='stretch_width')
                    [0] Button(button_type='primary', name='Output Directory')
                    [1] TextInput(sizing_mode='stretch_width', value='/glade/u/home/pearse')
                    [2] Modal(name='Select output directory')
                        [0